## Imports, Helpers, Parameters

### Imports

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from textwrap import wrap
import numpy as np
import os
import re
import matplotlib.font_manager as fm




### Parameters

In [26]:
main_color = "#3a5f83"

font_dir = r"C:\Users\teddy\Downloads\OAIPR\Technical\AEI Data Other\Lato"

# Loop through every file in the folder
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith(".ttf") and "lato" in font_file.lower():
        font_path = os.path.join(font_dir, font_file)
        fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "Lato"
plt.rcParams["font.weight"] = "normal"

palette = sns.color_palette("colorblind")
# Put once at the TOP of your notebook/script (or just tweak this line)
sns.set_context("notebook", font_scale=1.0)  # was 1.2; smaller = less crowded

chart_size = (17, 9)

## Automation By Task Completion

### Set Up DataFrame

In [27]:
# Load data (outside function)
ai_tasks = pd.read_csv("../data/tasks_final.csv")
all_tasks = pd.read_csv("../data/ratings_eco_2025.csv")
crosswalk_df = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")

# Calculate task completions
ai_tasks['task_comp_nat'] = ai_tasks['freq_mean_2025'] * ai_tasks['emp_tot_nat_2024']
ai_tasks['task_comp_ut'] = ai_tasks['freq_mean_2025'] * ai_tasks['emp_tot_ut_2024']

all_tasks['task_comp_nat'] = all_tasks['freq_mean'] * all_tasks['tot_emp_nat']
all_tasks['task_comp_ut'] = all_tasks['freq_mean'] * all_tasks['tot_emp_ut']


def create_df_imputing_method(ai_tasks_df, all_tasks_df, crosswalk_df) -> pd.DataFrame:
    """
    Merges AI-automatable task data with baseline task data, handling SOC code transitions
    from 2010 to 2019 and calculating automation percentages.
    
    This function:
    1. Aggregates AI task completions by 2010 SOC code
    2. Maps 2010 codes to 2019 codes using crosswalk
    3. Creates boolean flags for code/title changes
    4. Handles splits (one 2010 code → multiple 2019 codes)
    5. Aggregates baseline tasks to 2019 SOC level
    6. Merges datasets and calculates automation percentages
    
    Args:
        ai_tasks_df (pd.DataFrame): AI task data with columns:
            - soc_code_2010
            - title
            - task_comp_nat (freq × employment, national)
            - task_comp_ut (freq × employment, Utah)
            - freq_mean_2025
        all_tasks_df (pd.DataFrame): Baseline task data with columns:
            - soc_code_2019_untrimmed
            - title
            - task_comp_nat
            - task_comp_ut
            - freq_mean
            - tot_emp_nat, tot_emp_ut
            - major_occ_category, major_group_code
        crosswalk_df (pd.DataFrame): 2010 to 2019 SOC code crosswalk
    
    Returns:
        pd.DataFrame: Merged data with automation metrics including:
            - soc_code_2019
            - title
            - task_comp_nat/ut (baseline totals)
            - ai_task_comp_nat/ut (AI-automatable totals)
            - freq_sum_baseline (sum of baseline task frequencies)
            - freq_sum_ai (sum of AI task frequencies)
            - pct_automated_nat/ut
            - Boolean flags for code/title changes
    """
    
    # ===== Process AI Tasks =====
    # Aggregate to 2010 SOC level
    ai_agg = ai_tasks_df.groupby('soc_code_2010').agg({
        'title': 'first',
        'task_comp_nat': 'sum',
        'task_comp_ut': 'sum',
        'freq_mean_2025': 'sum',
    }).reset_index()
    
    # Merge with crosswalk
    ai_crosswalked = ai_agg.merge(
        crosswalk_df,
        left_on='soc_code_2010',
        right_on='O*NET-SOC 2010 Code',
        how='left'
    )
    
    # Create change indicator columns
    ai_crosswalked['code_changed'] = (
        ai_crosswalked['O*NET-SOC 2010 Code'] != ai_crosswalked['O*NET-SOC 2019 Code']
    )
    ai_crosswalked['title_changed'] = (
        ai_crosswalked['O*NET-SOC 2010 Title'] != ai_crosswalked['O*NET-SOC 2019 Title']
    )
    ai_crosswalked['code_or_title_changed'] = (
        ai_crosswalked['code_changed'] | ai_crosswalked['title_changed']
    )
    ai_crosswalked['code_and_title_changed'] = (
        ai_crosswalked['code_changed'] & ai_crosswalked['title_changed']
    )
    
    # Handle crosswalk splits (one 2010 → multiple 2019)
    ai_crosswalked['split_count'] = (
        ai_crosswalked.groupby('soc_code_2010')['soc_code_2010'].transform('count')
    )
    
    # Divide by split count to avoid double-counting
    ai_crosswalked['task_comp_nat_adj'] = (
        ai_crosswalked['task_comp_nat'] / ai_crosswalked['split_count']
    )
    ai_crosswalked['task_comp_ut_adj'] = (
        ai_crosswalked['task_comp_ut'] / ai_crosswalked['split_count']
    )
    ai_crosswalked['freq_2025_adj'] = (
        ai_crosswalked['freq_mean_2025'] / ai_crosswalked['split_count']
    )
    
    # Aggregate to 2019 SOC level
    ai_final = ai_crosswalked.groupby('O*NET-SOC 2019 Code').agg({
        'task_comp_nat_adj': 'sum',
        'task_comp_ut_adj': 'sum',
        'freq_2025_adj': 'sum',
        'code_changed': 'any',
        'title_changed': 'any',
        'code_or_title_changed': 'any',
        'code_and_title_changed': 'any'
    }).reset_index()
    
    ai_final.rename(columns={
        'O*NET-SOC 2019 Code': 'soc_code_2019',
        'task_comp_nat_adj': 'ai_task_comp_nat',
        'task_comp_ut_adj': 'ai_task_comp_ut',
        'freq_2025_adj': 'freq_sum_ai'
    }, inplace=True)
    
    # ===== Process Baseline Tasks =====
    baseline_agg = all_tasks_df.groupby('soc_code_2019_untrimmed').agg({
        'title': 'first',
        'task_comp_nat': 'sum',
        'task_comp_ut': 'sum',
        'freq_mean': 'sum',
        'importance': 'mean',
        'relevance': 'mean',
        'tot_emp_nat': 'first',
        'tot_emp_ut': 'first',
        'major_group_code': 'first',
        'major_occ_category': 'first'
    }).reset_index()
    
    baseline_agg.rename(columns={
        'soc_code_2019_untrimmed': 'soc_code_2019',
        'freq_mean': 'freq_sum_eco'
    }, inplace=True)
    
    # ===== Merge AI and Baseline =====
    merged = baseline_agg.merge(
        ai_final,
        on='soc_code_2019',
        how='left'
    )
    
    # Fill NaN for occupations without AI tasks
    merged[['ai_task_comp_nat', 'ai_task_comp_ut', 'freq_sum_ai']] = (
        merged[['ai_task_comp_nat', 'ai_task_comp_ut', 'freq_sum_ai']].fillna(0)
    )

    cols = ['code_changed', 'title_changed', 'code_or_title_changed', 'code_and_title_changed']
    merged[cols] = merged[cols].astype('boolean').fillna(False)

    
    # Calculate automation percentages
    merged['pct_automated_nat'] = (merged['ai_task_comp_nat'] / merged['task_comp_nat']) * 100
    merged['pct_automated_ut'] = (merged['ai_task_comp_ut'] / merged['task_comp_ut']) * 100

    # Calculate overall Utah share (across all occupations)
    total_ai_nat = merged['ai_task_comp_nat'].sum()
    total_ai_ut = merged['ai_task_comp_ut'].sum()
    overall_ut_share = total_ai_ut / total_ai_nat

    # Calculate Utah share for each occupation
    merged['ut_share_of_nat'] = merged['ai_task_comp_ut'] / merged['ai_task_comp_nat']

    # Calculate how much higher/lower than average
    merged['ut_concentration_index'] = merged['ut_share_of_nat'] / overall_ut_share



    # # Calculate overall Utah share (across all occupations)
    # total_ai_nat_emp = merged['tot_emp_nat'].sum()
    # total_ai_ut_emp = merged['tot_emp_ut'].sum()
    # overall_ut_share = total_ai_ut_emp / total_ai_nat_emp

    # # Calculate Utah share for each occupation
    # merged['ut_share_of_nat_emp'] = merged['tot_emp_ut'] / merged['tot_emp_nat']

    # # Calculate how much higher/lower than average
    # merged['ut_concentration_index_emp'] = merged['ut_share_of_nat_emp'] / overall_ut_share

    
    return merged


# Run the function
automation_tasks_imputed = create_df_imputing_method(ai_tasks, all_tasks, crosswalk_df)

automation_tasks_imputed["people_automated_nat"] = automation_tasks_imputed["pct_automated_nat"] * automation_tasks_imputed["tot_emp_nat"] / 100
automation_tasks_imputed["people_automated_ut"] = automation_tasks_imputed["pct_automated_nat"] * automation_tasks_imputed["tot_emp_ut"] / 100

### Top 15 % Occupation Automated

In [28]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'pct_automated_nat').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='pct_automated_nat', y='title_with_category', 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Most Automated Occupations by AI Task Coverage", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Percent of Occupation Tasks Automated (%)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 % Occupation Automated.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### % Major Occupational Category Automated

In [29]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("pct_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='pct_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Major Occupational Categories Automated by AI Task Coverage", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Percent of Category Tasks Automated (%)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "% Major Occupational Category Automated.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Workers Automated by Occupation (National)

In [30]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_nat').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_nat', y='title_with_category', 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Number of Workers Automated by Occupation by AI Task Coverage (National) ", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated (National)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Workers Automated by Occupation (National).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Workers Automated by Occupation (Utah)

In [31]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_ut').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_ut', y='title_with_category', 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Number of Workers Automated by Occupation by AI Task Coverage (Utah) ", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated (Utah)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Workers Automated by Occupation (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Workers Automated by Major Occupational Category (National)

In [32]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_nat": "sum",
}).reset_index()

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Automated by Major Occupational Category by AI Task Coverage (National)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Automated by Major Occupational Category (National).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Workers Automated by Major Occupational Category (Utah)

In [33]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_ut": "sum",
}).reset_index()

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Automated by Major Occupational Category by AI Task Coverage (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Automated by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)